# Validation

In [2]:
from pyspark.sql import SparkSession
from functools import reduce
from pyspark.sql.functions import col, year, to_date, lit, create_map, when
from pyspark.sql.functions import to_date, col, substring, concat_ws

import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")
import pessoal
from pessoal import *

spark = SparkSession.builder \
    .appName("SIM Processing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [23]:
from pyspark.sql.functions import regexp_replace

In [3]:
PATH = '/home/tatiane/repositorios/brazil-health-data-project/data_pipeline'
conversion_path = f"{PATH}/data/processed/sim/conversion/one_file/"
standardization_path = f"{PATH}/data/processed/sim/standardization/"

In [4]:
standardization = spark.read.parquet(standardization_path)
print('Qtd. registros:', standardization.count(), '| Quantidade de colunas: ', len(standardization.columns))

[Stage 1:===================================================>       (7 + 1) / 8]

Qtd. registros: 7600016 | Quantidade de colunas:  16


In [5]:
resumo = []
dfs = []

for ano in sorted(os.listdir(conversion_path)):
    caminho = os.path.join(conversion_path, ano)

    if os.path.isdir(caminho):
        df_ano = spark.read.parquet(caminho)

        resumo.append({
            "ano": ano,
            "registros": df_ano.count(),
            "colunas": len(df_ano.columns)
        })

        df_ano = df_ano.withColumn("ano", lit(ano))
        dfs.append(df_ano)

# visualizar resumo
for item in resumo:
    print(item)

{'ano': 'ano_2018', 'registros': 1316719, 'colunas': 88}
{'ano': 'ano_2019', 'registros': 1349801, 'colunas': 88}
{'ano': 'ano_2020', 'registros': 1556824, 'colunas': 88}
{'ano': 'ano_2021', 'registros': 1832649, 'colunas': 88}
{'ano': 'ano_2022', 'registros': 1544266, 'colunas': 88}


### Validation

- First validation
1. Quantidade de registros
2. Quantidade de colunas

In [6]:
df_sim_conversion = dfs[0]
for df in dfs[1:]:
    df_sim_conversion = df_sim_conversion.unionByName(df)

In [7]:
selected_cols = [
    "IDADE", "SEXO", "RACACOR", "ESTCIV",
    "CODMUNRES", "CODMUNOCOR",
    "DTOBITO", "DTNASC",
    "CAUSABAS", "CAUSABAS_O",
    "ASSISTMED", "NECROPSIA", "LOCOCOR"
]

df_sim_conversion = df_sim_conversion.select(*selected_cols)

In [8]:
mapping = {
    'IDADE':'idade',
    'SEXO':'sexo',
    'RACACOR':'raca_cor',
    'ESTCIV':'est_civ',
    'CODMUNRES':'cod_mun_res',
    'CODMUNOCOR':'cod_mun_ocor',
    'DTOBITO':'dt_obito',
    'DTNASC':'dt_nasc',
    'CAUSABAS':'causa_bas',
    'CAUSABAS_O':'causa_bas_o',
    'ASSISTMED':'assist_med',
    'NECROPSIA':'necropsia',
    'LOCOCOR':'local_ocor'
}

In [9]:
def rename_columns(df_sim_conversion, mapping):
    return reduce(
        lambda df_sim_conversion, col: df_sim_conversion.withColumnRenamed(col, mapping[col]),
        mapping,
        df_sim_conversion
    )

In [12]:
df_sim_conversion = rename_columns(df_sim_conversion, mapping)

In [13]:
conversion = df_sim_conversion.toDF(*[c.lower() for c in df_sim_conversion.columns])
pessoal.completudeSchema(conversion)

Qtd. registros: 7600259 | Quantidade de colunas:  13
root
 |-- idade: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- est_civ: string (nullable = true)
 |-- cod_mun_res: string (nullable = true)
 |-- cod_mun_ocor: string (nullable = true)
 |-- dt_obito: string (nullable = true)
 |-- dt_nasc: string (nullable = true)
 |-- causa_bas: string (nullable = true)
 |-- causa_bas_o: string (nullable = true)
 |-- assist_med: string (nullable = true)
 |-- necropsia: string (nullable = true)
 |-- local_ocor: string (nullable = true)



In [14]:
pessoal.completudeSchema(standardization)

Qtd. registros: 7600016 | Quantidade de colunas:  16
root
 |-- causa_bas: string (nullable = true)
 |-- cod_mun_res: string (nullable = true)
 |-- cod_mun_ocor: string (nullable = true)
 |-- causa_bas_o: string (nullable = true)
 |-- dt_nasc: date (nullable = true)
 |-- dt_obito: date (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- est_civ: string (nullable = true)
 |-- assist_med: string (nullable = true)
 |-- necropsia: string (nullable = true)
 |-- local_ocor: string (nullable = true)
 |-- idade: double (nullable = true)
 |-- faixa_etaria: string (nullable = true)
 |-- nm_doenca: string (nullable = true)
 |-- houve_ajuste_causa: string (nullable = true)



In [17]:
# 1. Transforma as listas de colunas em conjuntos (sets)
colunas_conv = set(conversion.columns)
colunas_pad = set(standardization.columns)

# 2. Encontra as colunas que estão em AMBOS os DataFrames
colunas_em_comum = list(colunas_conv.intersection(colunas_pad))

# 3. Exibe o resultado
print(f"Variáveis em comum ({len(colunas_em_comum)}): {colunas_em_comum}")

# 4. Retorna o que é diferente
colunas_diferentes = list(colunas_conv.symmetric_difference(colunas_pad))

print(f"Total de colunas diferentes: {len(colunas_diferentes)}")
print(f"Colunas diferentes: {colunas_diferentes}")

Variáveis em comum (13): ['local_ocor', 'cod_mun_res', 'dt_nasc', 'causa_bas', 'dt_obito', 'raca_cor', 'necropsia', 'sexo', 'causa_bas_o', 'idade', 'assist_med', 'cod_mun_ocor', 'est_civ']
Total de colunas diferentes: 3
Colunas diferentes: ['nm_doenca', 'faixa_etaria', 'houve_ajuste_causa']


In [35]:
7600259 - 7600016

243